# Breakout 1B · Sprint 1 · build the rule book yourself, then decide which rule to remove

Every line of the **random draw of months 19 to 36** (the honest test: nobody chose those lines because of a rule) shows the rules it triggers, the lane the rule book gives it, the control done and its result. **R2 is worked as the example.** You compute R1, R3 and R4 the same way, then decide keep, revise or retire for each.

Run each cell with **Shift + Enter**. Fill only the cells marked **YOUR TURN**; replace each `...`. A line starting with ✓ means your number is right. Stuck? `1B_compute_worked.xlsx` shows every formula, and `1B_lab.xlsx` has every number already computed: use them to check, after trying.

*Published customs datasets record an irregularity on 1 to 8 lines in 100; in this lab the random draw finds 3 in 100.*

In [ ]:
# Run this cell first (click it, then Shift + Enter). It loads the lab data.
import hashlib, json, os
from pathlib import Path
import numpy as np, pandas as pd

# The folder that holds features_lines.csv. Change it here if your data is elsewhere.
CANDIDATES = [os.environ.get('WORKSHOP_DATA', ''), 'data', '../data', '/workspace/data', '.']
DATA = next(Path(c) for c in CANDIDATES if c and (Path(c) / 'features_lines.csv').exists())
lines = pd.read_csv(DATA / 'features_lines.csv', low_memory=False)
lines['found'] = (lines.outcome == 'finding').astype(int)
print('Data folder:', DATA, '·', f'{len(lines):,}', 'declaration lines')

def _h(v, nd):
    return hashlib.sha256(f'{round(float(v), nd):.{nd}f}'.encode()).hexdigest()[:16]
def check(name, value, expected, nd):
    ok = value is not None and value is not ... and _h(value, nd) == expected
    num = isinstance(value, (int, float, np.integer, np.floating))
    shown = (f'{value:.{nd}f}' if 'code' in name else f'{value:,.{nd}f}') if num else str(value)
    print(('✓ ' if ok else '✗ not yet: ') + name + ' = ' + shown)


## The lines: rules triggered, lane, control and result

In [ ]:
tariff = pd.read_csv(DATA / 'tariff.csv')[['hs8', 'static_reference_2022']]
draw = lines[(lines.random_draw == 1) & (lines.period_month >= 19)].merge(tariff, on='hs8', how='left')
usd = draw.value_per_kg / 25   # the 2022 list is in dollars, 25 LCU to the dollar
draw['R1'] = (draw.new_importer == 1).astype(int)
draw['R2'] = (usd < 0.7 * draw.static_reference_2022).astype(int)
draw['R3'] = ((draw.chapter.isin([22, 24, 85, 87]) | (draw.hs6 == 170199)) & draw.office.isin(['AIR2', 'LAND1'])).astype(int)
draw['R4'] = ((draw.preference_claim == 'FTA') & draw.origin.isin(['MYS', 'IDN'])).astype(int)
draw['n_rules'] = draw[['R1', 'R2', 'R3', 'R4']].sum(axis=1)
draw['lane'] = np.where(draw.R2 == 1, 'red', np.where(draw[['R1', 'R3', 'R4']].sum(axis=1) > 0, 'yellow', 'green'))
base = draw.found.mean()   # the random draw's own hit rate
print(f'{len(draw):,} lines, {draw.found.sum()} findings, base {base:.1%}')
draw[['line_id', 'period_month', 'R1', 'R2', 'R3', 'R4', 'n_rules', 'lane', 'control_type', 'found', 'finding_type', 'duty_recovered_lcu']].head()

## R2, the worked example

Read each line: it is the formula you will reuse.

In [ ]:
flag = draw['R2'] == 1
flagged = flag.sum()                                   # lines the rule flags
findings = (flag & (draw.found == 1)).sum()            # among them, lines that found something
hit_rate = findings / flagged
lift = hit_rate / base                                 # above 1 beats chance; 2 or more pays
share = flagged / len(draw)                            # close to the share of traffic; budget 5%
new = (flag & (draw.n_rules == 1)).sum() / flagged      # flags no other rule raises
duty = draw.loc[flag, 'duty_recovered_lcu'].sum() / flagged
print(f'R2: {flagged} lines, {findings} findings, hit rate {hit_rate:.1%}, lift {lift:.2f}, share {share:.1%}, new {new:.0%}, duty per line {duty:,.0f}')

## YOUR TURN: R1, R3 and R4

**Hint 1:** copy each line of the R2 cell; only the rule changes.

<details><summary>Hint 2</summary>

```python
flagged = flag.sum()
findings = (flag & (draw.found == 1)).sum()
hit_rate = findings / flagged
lift = hit_rate / base
share = flagged / len(draw)
new = (flag & (draw.n_rules == 1)).sum() / flagged
duty = draw.loc[flag, 'duty_recovered_lcu'].sum() / flagged
```
</details>

In [ ]:
# YOUR TURN: the same for R1, then change RULE to 'R3' and 'R4' and run the cell again
RULE = 'R1'
flag = draw[RULE] == 1
flagged = ...     # lines the rule flags
findings = ...    # among them, lines where the control found something
hit_rate = ...    # findings ÷ flagged
lift = ...        # hit_rate ÷ base
share = ...       # flagged ÷ all lines of the draw
new = ...         # share of its flags that no other rule raises (draw.n_rules == 1)
duty = ...        # duty recovered on its flagged lines ÷ flagged

In [ ]:
E = {"R1": {"flagged": "38b2d03f3256502b", "findings": "2c624232cdd22177", "hit_rate": "9a606d72466d36f2", "lift": "64ee787d2a1eded8", "share": "69b94851d8dca555", "new": "d1318066c8694e30", "duty": "d2572ef356144993"}, "R2": {"flagged": "94ad0b5b0a595b7e", "findings": "05ada863a4cf9660", "hit_rate": "8f7c203b09edb33e", "lift": "caac385a527f880b", "share": "f31729b0155a55c2", "new": "f296a60798b8990c", "duty": "b786f156b551a7b9"}, "R3": {"flagged": "5092c37bcbc9f0fb", "findings": "8527a891e2241369", "hit_rate": "3f50e94117e6686f", "lift": "89e0e9d0a617da9c", "share": "34edf0028b4ae0c0", "new": "376a082b389e2be7", "duty": "d346608c680087a8"}, "R4": {"flagged": "0ef962215cc05578", "findings": "98010bd9270f9b10", "hit_rate": "9cbac2294a3f6903", "lift": "a53b65946f6a4cdb", "share": "cf07eaf1fef3deb4", "new": "7a00f43870dded46", "duty": "fe3af2f3315ce61a"}, "base": "4a226119686fa6d8"}
for name, v, key, nd in [('lines flagged', flagged, 'flagged', 0), ('findings', findings, 'findings', 0), ('hit rate', hit_rate, 'hit_rate', 4), ('lift', lift, 'lift', 2),
                         ('share of the draw', share, 'share', 4), ('new to the book', new, 'new', 3), ('duty per line checked', duty, 'duty', 0)]:
    check(RULE + ' ' + name, v, E[RULE][key], nd)

## If you have time: write one rule yourself

**Hint 1:** a test looks like `draw.new_importer == 1`; combine tests with `&` and `|`, each in brackets; `.isin([...])` tests a list.

<details><summary>Hint 2</summary>

```python
mine = (draw.value_per_kg / 25) < 0.7 * draw.static_reference_2022   # R2
```
</details>

In [ ]:
# YOUR TURN (if you have time): write one rule yourself as a test, and compare its count with the rule's column
mine = ...
print(mine.sum(), draw['R2'].sum())

## Then

Decide, rule by rule: **keep, revise or retire**, with the number that decided. A lift under 1.5 rarely pays; a share above 5% makes a queue; few flags new to the book means another rule already does the job. Lock on the site: once your numbers are entered, the site shows its own table beside yours.